# XGBoost

**The big picture:** the data has 683 breast-tissue samples. Each one has 9 scores from 1 to 10 describing how the cells look under a microscope (size, shape, thickness and so on). The goal is to predict whether the tumour is **benign** (harmless) or **malignant** (cancer).

**XGBoost** builds **many small decision trees, each one fixing the previous ones' mistakes**. The first tree makes rough guesses. The second tree focuses on where the first one went wrong, the third on what is still wrong, and so on. Added together, lots of weak guessers become one strong one. This step-by-step fixing is called *gradient boosting*. XGBoost ("eXtreme Gradient Boosting") is a fast, popular version of it that often wins competitions on table-shaped data.

### Before you start
- **Read first:** **decision_tree_classification** and **random_forest_classification** (XGBoost is also built from many trees), plus **k_fold_cross_validation** (used at the end to check the model).
- **New words in this notebook:** *gradient boosting*, *XGBoost*, *weak learner*, *LabelEncoder*, *false negative*. Look them up in the [glossary](../../../GLOSSARY.md) if you get stuck.

## Importing the libraries

- **pandas** reads and handles tables, like Excel in Python.
- **numpy** does fast maths on lists of numbers.
- **matplotlib** draws charts. It is imported out of habit but not used: this notebook has no plots.

In [0]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

## Importing the dataset

- Reads the CSV file into a table of **683 samples**.
- **X** = the 9 cell scores (Clump Thickness, Uniformity of Cell Size, Uniformity of Cell Shape, Marginal Adhesion, Single Epithelial Cell Size, Bare Nuclei, Bland Chromatin, Normal Nucleoli, Mitoses).
- **y** = *Class*: **2 = benign** (444 samples), **4 = malignant** (239 samples). This is what we want to **predict**.
- `iloc[:, 1:-1]` skips the first column, *Sample code number*. It is just an ID label (like a ticket number), so it says nothing about the cells and would only confuse the model.
- `LabelEncoder` renames the classes **2 → 0 (benign)** and **4 → 1 (malignant)**. **Why?** XGBoost expects the classes to be numbered 0, 1, 2, … and stops with an error if they are 2 and 4.

In [0]:
dataset = pd.read_csv('Data.csv')
X = dataset.iloc[:, 1:-1].values
y = dataset.iloc[:, -1].values
from sklearn.preprocessing import LabelEncoder
y = LabelEncoder().fit_transform(y)  # 2 (benign) -> 0, 4 (malignant) -> 1

## Splitting the dataset into the Training set and Test set

- **80% of the samples (546)** go into the *training set*. The model learns from these.
- **20% (137 samples)** go into the *test set*, held back to check the model afterwards.
- `random_state = 0` makes the shuffle come out the same every time.

There is **no feature scaling** in this notebook. **Why?** Decision trees only ask yes/no questions like "is Clump Thickness above 5?". The size of the numbers doesn't matter to them, so scaling isn't needed.

In [0]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 0)

## Training XGBoost on the Training set

- `XGBClassifier()` creates the model with all **default settings** (for example, 100 trees, each limited to a handful of questions deep).
- `.fit()` builds the trees one after another, each one trained to correct the mistakes left by the trees before it.

The notebook has no saved output, so run it to see the results.

In [0]:
from xgboost import XGBClassifier
classifier = XGBClassifier()
classifier.fit(X_train, y_train)

## Making the Confusion Matrix

- `predict` guesses benign / malignant for the 137 test samples.
- The **confusion matrix** is a 2×2 table: rows = the real class (0 = benign, then 1 = malignant), columns = the guessed class (same order).
  - Top-left: benign correctly called benign.
  - Bottom-right: malignant correctly called malignant.
  - Top-right: benign wrongly called malignant (a false alarm).
  - Bottom-left: malignant wrongly called benign (a **missed cancer**, the most serious mistake here).
- `accuracy_score` = correct guesses ÷ 137.

The notebook has no saved output, so run the cell to see the numbers. XGBoost usually scores very highly on this dataset.

In [0]:
from sklearn.metrics import confusion_matrix, accuracy_score
y_pred = classifier.predict(X_test)
cm = confusion_matrix(y_test, y_pred)
print(cm)
accuracy_score(y_test, y_pred)

## Applying k-Fold Cross Validation

- A single test score can be lucky or unlucky, so the model also gets **10 mini exams**.
- `cv = 10` cuts the **546 training samples into 10 groups** (about 55 each). The model is trained 10 times, each time tested on a different group and trained on the other 9.
- The first print is the **average accuracy** of the 10 exams: the best estimate of how good the model really is.
- The second print is the **standard deviation**: how much the 10 scores jump around. Small = consistently good; large = the result depends a lot on which samples it sees.

**In one sentence:** load 683 breast-tissue samples, train XGBoost (many small trees, each fixing the previous ones' mistakes) to tell benign from malignant, then check it with a confusion matrix on 137 unseen samples and with 10-fold cross validation.

In [0]:
from sklearn.model_selection import cross_val_score
accuracies = cross_val_score(estimator = classifier, X = X_train, y = y_train, cv = 10)
print("Accuracy: {:.2f} %".format(accuracies.mean()*100))
print("Standard Deviation: {:.2f} %".format(accuracies.std()*100))

## Common mistakes to avoid
- **Feeding XGBoost labels like 2 and 4.** Recent XGBoost versions expect classes numbered 0, 1, 2, … and stop with an error otherwise. This notebook fixes it with `LabelEncoder` (2 → 0 benign, 4 → 1 malignant); remember the confusion matrix rows and columns are now in that 0, 1 order.
- **Leaving an ID column in the inputs.** *Sample code number* is just a label, but a model can still latch onto it by accident and learn nonsense. That's why `iloc[:, 1:-1]` skips the first column; always drop IDs, row numbers and similar columns.
- **Judging the model only by accuracy on a medical problem.** Missing a cancer (malignant called benign, the bottom-left of the confusion matrix) is far worse than a false alarm. Read the confusion matrix, not just the single accuracy number.
- **Cranking up the number of trees and their depth to chase a perfect score.** Boosting keeps fixing mistakes, so with too many or too deep trees it starts memorising noise in the training data (overfitting). Watch the cross-validation score, not the training score, when changing settings.

## Try this yourself
Change one thing, rerun the notebook, and see what happens.

1. In the training cell, change `XGBClassifier()` to `XGBClassifier(n_estimators = 5)` and rerun the cells below. What to notice: with only 5 trees there are fewer rounds of fixing mistakes; compare the confusion matrix, accuracy and cross-validation numbers with the default (100 trees).
2. In the training cell, try `XGBClassifier(max_depth = 1)` (each tree asks just one yes/no question) and then `XGBClassifier(max_depth = 10)`. What to notice: even single-question trees can do well once many are added together, which is the whole idea of boosting weak guessers.
3. Add a new cell after training: `print(dict(zip(dataset.columns[1:-1], classifier.feature_importances_.round(3))))`. What to notice: which of the 9 cell scores the trees rely on most; often a few features carry most of the weight.
4. Add a new cell to compare with an earlier model: `from sklearn.ensemble import RandomForestClassifier` then `rf = RandomForestClassifier(n_estimators = 100, random_state = 0)` and `print(cross_val_score(rf, X_train, y_train, cv = 10).mean())` (run it after the k-Fold cell so `cross_val_score` is imported). What to notice: how close the random forest's average accuracy is to XGBoost's on this fairly easy dataset.

## Check your understanding
Try to answer in your head first, then click to reveal.

**1. How is XGBoost different from a random forest, if both use many trees?**

<details><summary>Show answer</summary>

A random forest builds its trees independently and lets them vote. XGBoost builds trees one after another, and each new tree focuses on fixing the mistakes the earlier trees still make. That step-by-step fixing is called gradient boosting.

</details>

**2. Why is there no feature scaling in this notebook?**

<details><summary>Show answer</summary>

Trees only ask yes/no questions such as "is Clump Thickness above 5?". Whether the numbers are big or small doesn't change the answer to those questions, so scaling isn't needed.

</details>

**3. When you run the confusion matrix cell, which cell of the 2×2 table holds the most dangerous mistakes, and why?**

<details><summary>Show answer</summary>

The bottom-left cell. Rows are the real class and columns the guess, in the order 0 (benign), 1 (malignant), so bottom-left means the tumour was really malignant but the model said benign: a missed cancer. A false alarm (top-right) leads to extra tests; a miss could mean no treatment.

</details>

**4. What would happen if you removed the `LabelEncoder` line and kept y as 2 and 4?**

<details><summary>Show answer</summary>

Recent versions of XGBoost would stop with an error at `classifier.fit`, complaining that the class labels are not 0 and 1. The encoder simply renames 2 → 0 and 4 → 1 so XGBoost accepts them.

</details>

---
📚 [Learning path](../../../LEARNING_PATH.md) · [Glossary](../../../GLOSSARY.md)